# error analysis of model predicition

In [ ]:
from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve().parents[0]
PROMPTING_PATH = PROJECT_ROOT / "outputs" / "predictions" / "prompting"

models = ["3B", "7B"]
modes = ["zero_shot", "five_shot"]
targets = ["english", "de", "sv"]

all_dfs_prompting_values = []
all_dfs_prompting_metrics = []

for mode in modes:
    for model in models:
        for target in targets:
            file_path = PROMPTING_PATH / mode / f"test_{model}_{target}_predictions.csv"

            if file_path.exists():
                df = pd.read_csv(file_path)
                # Tag the rows so you know which condition generated them
                df["model"] = model
                df["mode"] = mode
                df["target"] = target
                all_dfs_prompting_values.append(df)
            else:
                print(f"Warning: Could not find {file_path}")

# This master DataFrame is perfect for groupby operations and Seaborn plotting
master_df = pd.concat(all_dfs_prompting_values, ignore_index=True)

# load all the metrics for the prompting experiments
for mode in modes:
    for model in models:
        for target in targets:
            file_path = PROMPTING_PATH / mode / f"test_{model}_{target}_metrics.csv"

            if file_path.exists():
                df = pd.read_csv(file_path)
                # Tag the rows so you know which condition generated them
                df["model"] = model
                df["mode"] = mode
                df["target"] = target
                all_dfs_prompting_metrics.append(df)
            else:
                print(f"Warning: Could not find {file_path}")



In [6]:
# Create a wide format to compare predictions directly across models for the same text
comparison_df = master_df.pivot_table(
    index=["alignment_id", "true_labels"], 
    columns=["model", "mode", "target"], 
    values="normalized_prediction", 
    aggfunc="first"
)

# Example: View the side-by-side predictions for the first 5 sentences
display(comparison_df.head())

model                                                                                3B  \
mode                                                                          five_shot   
target                                                                               de   
alignment_id                     true_labels                                              
en/1953/47677/5041695.xml.gz#290 ['surprise']                                   ['joy']   
en/1953/47677/5041695.xml.gz#351 ['disgust']              ['disgust', 'joy', 'sadness']   
en/1953/47677/5041695.xml.gz#355 ['fear']                                   ['sadness']   
en/1953/47677/5041695.xml.gz#387 ['sadness', 'surprise']                    ['disgust']   
en/1953/47677/5041695.xml.gz#498 ['surprise']                          ['anticipation']   

model                                                                  \
mode                                                                    
target                                                   english   sv   
alignment_id                     true_labels                            
en/1953/47677/5041695.xml.gz#290 ['surprise']                NaN  NaN   
en/1953/47677/5041695.xml.gz#351 ['disgust']                 NaN  NaN   
en/1953/47677/5041695.xml.gz#355 ['fear']                    NaN  NaN   
en/1953/47677/5041695.xml.gz#387 ['sadness', 'surprise']     NaN  NaN   
en/1953/47677/5041695.xml.gz#498 ['surprise']                NaN  NaN   

model                                                                       \
mode                                                             zero_shot   
target                                                                  de   
alignment_id                     true_labels                                 
en/1953/47677/5041695.xml.gz#290 ['surprise']                           []   
en/1953/47677/5041695.xml.gz#351 ['disgust']                            []   
en/1953/47677/5041695.xml.gz#355 ['fear']                               []   
en/1953/47677/5041695.xml.gz#387 ['sadness', 'surprise']                []   
en/1953/47677/5041695.xml.gz#498 ['surprise']             ['anticipation']   

model                                                                  
mode                                                                   
target                                                   english   sv  
alignment_id                     true_labels                           
en/1953/47677/5041695.xml.gz#290 ['surprise']                NaN  NaN  
en/1953/47677/5041695.xml.gz#351 ['disgust']                 NaN  NaN  
en/1953/47677/5041695.xml.gz#355 ['fear']                    NaN  NaN  
en/1953/47677/5041695.xml.gz#387 ['sadness', 'surprise']     NaN  NaN  
en/1953/47677/5041695.xml.gz#498 ['surprise']                NaN  NaN